# 04 — Retrieval: Two-Tower

**Owner:** Person C · **Course session:** S3 (Retrieval at Scale) · **25% of grade with Notebook 03 · heaviest component**

Trains a two-tower retrieval model from scratch (`src/models/two_tower.py`) with in-batch
negatives + softmax cross-entropy, tunes on **val only**, evaluates against Popularity and
MF-BPR, and saves checkpoints for Notebook 05 (LightGBM) and the webapp to reload.

> **Colab setup:** Runtime → Change runtime type → **GPU (T4)**. This is the heaviest required
> component (10-15h across the team) — start it early. Mount Drive and save
> `data/two_tower_checkpoint.pt` there right after training so a disconnect doesn't cost a
> re-run.

**Before the defense, every team member must be able to explain (1) why in-batch negatives and
(2) why softmax cross-entropy over the batch** — both called out explicitly in
`src/models/two_tower.py`'s docstring as defense questions.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # macOS: torch+faiss both bundle OpenMP and abort on double-init otherwise
import sys, time, json
from pathlib import Path
sys.path.insert(0, ".." if Path("../src").exists() else ".")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src import config as C
from src.models.two_tower import TwoTower, make_history_batches, export_item_embeddings, user_vector
from src.retrieval.faiss_index import EmbeddingIndex
from src.eval.harness import evaluate_index, two_tower_user_vecs
from src.data.split import ground_truth_from

plt.rcParams["figure.dpi"] = 110
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

DATA = Path("../data") if Path("../data").exists() else Path("data")
train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "val.parquet")
test = pd.read_parquet(DATA / "test.parquet")
ids = json.load(open(DATA / "id_encoders.json"))
user_ids, item_ids = ids["user_ids"], ids["item_ids"]
uid2u = {u: i for i, u in enumerate(user_ids)}
n_users, n_items = len(user_ids), len(item_ids)
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}  n_users={n_users:,}  n_items={n_items:,}")

comparison_results = pd.read_csv(DATA / "comparison_results.csv", index_col=0)
comparison_results

## Id spaces (same convention as Notebook 03)

Encoded ints (`u`/`i`) only to index into the model's embedding tables / build a history
sequence; everything eval-related (`ground_truth_from`, `seen_items`, `evaluate`, FAISS
labels) stays in raw string id space. `n_items + 1` reserves the trailing `<UNK>` row from
`encode_ids`, though in practice this eval never needs it: cold (train-unseen) test users are
simply excluded — the whole point of the cold-start analysis in Notebook 05 is to quantify how
much that costs, not to paper over it here.

## Training loop (with loss history) + a batched helper

`make_history_batches` sorts each user's train interactions by `timestamp` and, for every
`(user, target)` pair, uses only *earlier* items as history — the model never sees the target
or anything after it. This duplicates `src.models.two_tower.train_two_tower`'s loop with two
additions: a `temperature` argument (needed for the val-only tuning grid below — the shipped
`train_two_tower` hardcodes the model default) and loss history for plotting.

In [ ]:
def train_two_tower_with_history(train_df, n_items_total, dim, temperature, lr, epochs,
                                 batch_size, max_hist, device):
    model = TwoTower(n_items_total, dim, temperature=temperature).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for ep in range(epochs):
        model.train(); running = n = 0
        for hist, mask, pos in make_history_batches(train_df, max_hist, batch_size, seed=ep):
            hist = torch.as_tensor(hist, device=device)
            mask = torch.as_tensor(mask, device=device)
            pos = torch.as_tensor(pos, device=device)
            opt.zero_grad()
            loss = model.in_batch_loss(hist, mask, pos)
            loss.backward(); opt.step()
            running += loss.item(); n += 1
        history.append(running / max(n, 1))
        print(f"epoch {ep+1}/{epochs}  loss={history[-1]:.4f}")
    return model, history


def two_tower_retrieval_metrics(model, eval_df, train_df, catalog_size, k_values, max_hist):
    """Builds each evaluated user's vector from their TRAIN-period history (not
    test-period) so this is a fair, leakage-free train->predict-future-test comparison
    against MF-BPR/Popularity. (At serving time the API does the opposite — runs the user
    tower over TEST-period history, i.e. the user's most recent real activity, to predict
    what they'll want *next*. Both are correct; they answer different questions.)"""
    gt = ground_truth_from(eval_df, positive_only=True)
    users = [uid for uid in gt if uid in uid2u]                     # drop cold (train-unseen) users
    train_hist = train_df.sort_values("timestamp").groupby("u")["i"].agg(list).to_dict()
    histories = [train_hist.get(uid2u[uid], []) for uid in users]
    uvecs = two_tower_user_vecs(model, histories, max_hist=max_hist, device=device_of(model))
    item_emb = export_item_embeddings(model, n_items + 1, device=device_of(model))[:n_items]
    index = EmbeddingIndex(item_emb, item_ids)
    return evaluate_index(index, uvecs, users, gt, train_df, catalog_size, k_values)


def device_of(model):
    return next(model.parameters()).device

## Hyperparameter search (val only)

Grid over `temperature` / `dim` / `max_hist`, each run a single short epoch — enough to rank
configs directionally, not to converge. **Never touches `test` here.**

In [ ]:
grid = [
    dict(temperature=0.05, dim=64, max_hist=20),
    dict(temperature=0.10, dim=64, max_hist=20),
    dict(temperature=0.05, dim=64, max_hist=10),
    dict(temperature=0.05, dim=32, max_hist=20),
]
search_rows = []
for cfg in grid:
    m, _ = train_two_tower_with_history(train, n_items + 1, cfg["dim"], cfg["temperature"],
                                        lr=1e-3, epochs=1, batch_size=1024,
                                        max_hist=cfg["max_hist"], device=DEVICE)
    _, m20 = two_tower_retrieval_metrics(m, val, train, catalog_size=n_items,
                                         k_values=(20,), max_hist=cfg["max_hist"])
    search_rows.append({**cfg, "val_Recall@20": m20["Recall@20"]})

search_df = pd.DataFrame(search_rows).sort_values("val_Recall@20", ascending=False)
search_df

**Pick the best row above** and train the final model longer (`EPOCHS=10`, matching
`src/models/two_tower.py`'s own default — the search only had to rank configs).

In [ ]:
BEST = search_df.iloc[0][["temperature", "dim", "max_hist"]].to_dict()
BEST["dim"] = int(BEST["dim"]); BEST["max_hist"] = int(BEST["max_hist"])
EPOCHS = 10
print("training with:", BEST)

t0 = time.time()
model, loss_history = train_two_tower_with_history(train, n_items + 1, BEST["dim"],
                                                    BEST["temperature"], lr=1e-3, epochs=EPOCHS,
                                                    batch_size=1024, max_hist=BEST["max_hist"],
                                                    device=DEVICE)
print(f"trained in {time.time()-t0:.0f}s")

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(range(1, len(loss_history) + 1), loss_history, color="#a05d3b", marker="o")
ax.set_xlabel("epoch"); ax.set_ylabel("mean in-batch softmax CE loss"); ax.set_title("Two-tower training loss")
plt.tight_layout(); plt.show()

## Retrieval evaluation (test) — full comparison table

Popularity vs MF-BPR vs Two-tower, side by side — required in both `README.md` and
`ANALYSIS.md`.

In [ ]:
recs, tt_metrics = two_tower_retrieval_metrics(model, test, train, catalog_size=n_items,
                                               k_values=C.K_VALUES, max_hist=BEST["max_hist"])
tt_metrics

In [ ]:
comparison_results.loc["Two-tower"] = {k: v for k, v in tt_metrics.items() if k != "n_users_scored"}
comparison_results.round(4)

## Interpretation

Two-tower should edge out MF-BPR on Recall/NDCG — it conditions on a user's actual recent
history (not just a fixed learned embedding), and in-batch negatives give a denser training
signal per step than BPR's one-negative-at-a-time sampling. If it doesn't win outright here,
note that honestly — the dataset is small and sparse at this sample size (see Notebook 01),
and "best" is whatever the numbers above actually say, not a foregone conclusion. Either way,
Two-tower is what gets deployed (next section) regardless of which model wins this table,
because of how it's built, not because it won.

## Why Two-tower serves in production regardless of which model wins above

`api/recommender.py`'s `_load_tower` loads a **TorchScripted user tower** and forward-passes
a user's live TEST-period history through it at request time — that's how the API handles a
user whose latest activity the model never trained on without needing to retrain. MF-BPR has
no equivalent: its "user vector" is one fixed row in a lookup table, learned only for the
exact users seen during training, with no way to incorporate a user's later activity short of
retraining from scratch. So MF-BPR (Notebook 03) is a required, independently defensible
implementation and comparison-table entry, but the two-tower model here is what
`scripts/export_artifacts.py` scripts and what the API actually serves — this is a real
architectural constraint, not a preference.

## Precompute similar items (for "Similar items" and "Because you liked X")

Static — every item's nearest neighbours in embedding space, computed once here and served
straight from JSON at request time (`api/recommender.py`'s `.similar()` never touches FAISS
live).

In [ ]:
item_emb_full = export_item_embeddings(model, n_items + 1, device=DEVICE)
item_emb_real = item_emb_full[:n_items]                       # drop the <UNK> row
index = EmbeddingIndex(item_emb_real, item_ids)

t0 = time.time()
similar_map = {iid: index.similar_items(iid, n=10) for iid in item_ids}
print(f"precomputed similar_items for {len(similar_map):,} items in {time.time()-t0:.0f}s")
list(similar_map.items())[:2]

## Save checkpoint + embeddings for Notebook 05

Each notebook here is its own Colab session, so we checkpoint everything Notebook 05 needs to
reload rather than assume shared memory — same reasoning as freezing the split in Notebook 01.
Save these to Drive in the real run so a disconnect doesn't cost the 10-15h of work.

In [ ]:
torch.save({
    "state_dict": model.state_dict(),
    "n_items_total": n_items + 1,
    "dim": BEST["dim"],
    "temperature": BEST["temperature"],
    "max_hist": BEST["max_hist"],
}, DATA / "two_tower_checkpoint.pt")

np.save(DATA / "item_emb.npy", item_emb_real)
json.dump(similar_map, open(DATA / "similar_items.json", "w"))
comparison_results.to_csv(DATA / "comparison_results.csv")
print("saved -> data/{two_tower_checkpoint.pt, item_emb.npy, similar_items.json, comparison_results.csv}")